# Chapter 14 — Databases

SQL is SQL, so Python's job here is to host a real database. We use the built-in `sqlite3` module with an in-memory database, so every query in this notebook actually runs.

Topics: implicit vs. explicit joins, normalized vs. denormalized schemas, the core SQL statements, and small database design.

### Schema used by 14.1–14.3
Apartment complexes contain buildings, buildings contain apartments, tenants rent apartments (many-to-many), and apartments have maintenance requests.

In [1]:
import sqlite3

db = sqlite3.connect(":memory:")
db.executescript("""
CREATE TABLE Complexes  (ComplexID INTEGER PRIMARY KEY, ComplexName TEXT);
CREATE TABLE Buildings  (BuildingID INTEGER PRIMARY KEY, ComplexID INTEGER REFERENCES Complexes,
                         BuildingName TEXT, Address TEXT);
CREATE TABLE Apartments (AptID INTEGER PRIMARY KEY, UnitNumber TEXT, BuildingID INTEGER REFERENCES Buildings);
CREATE TABLE Tenants    (TenantID INTEGER PRIMARY KEY, TenantName TEXT);
CREATE TABLE AptTenants (TenantID INTEGER REFERENCES Tenants, AptID INTEGER REFERENCES Apartments);
CREATE TABLE Requests   (RequestID INTEGER PRIMARY KEY, Status TEXT, AptID INTEGER REFERENCES Apartments,
                         Description TEXT);

INSERT INTO Complexes VALUES (1, 'Maple Grove');
INSERT INTO Buildings VALUES (10, 1, 'North', '1 Maple St'), (11, 1, 'South', '2 Maple St'), (12, 1, 'East', '3 Maple St');
INSERT INTO Apartments VALUES (100, '1A', 10), (101, '1B', 10), (110, '2A', 11), (111, '2B', 11), (120, '3A', 12);
INSERT INTO Tenants VALUES (1, 'Ana'), (2, 'Ben'), (3, 'Cleo');
INSERT INTO AptTenants VALUES (1, 100), (1, 110), (2, 101), (3, 111), (3, 120), (3, 100);
INSERT INTO Requests VALUES
  (1, 'Open', 100, 'leak'), (2, 'Closed', 100, 'paint'), (3, 'Open', 110, 'heater'),
  (4, 'Open', 111, 'door'), (5, 'Open', 101, 'window');
""")


def q(sql: str, *args) -> list[tuple]:
    return db.execute(sql, args).fetchall()


print(q("SELECT name FROM sqlite_master WHERE type='table'"))

[('Complexes',), ('Buildings',), ('Apartments',), ('Tenants',), ('AptTenants',), ('Requests',)]


## 14.1 Multiple Apartments
List every tenant who rents more than one apartment. Group `AptTenants` by tenant with `HAVING COUNT(*) > 1`, then join to get the names. The inner query needs no join, so it stays cheap.

In [2]:
rows = q("""
SELECT TenantName
FROM Tenants
JOIN (SELECT TenantID FROM AptTenants GROUP BY TenantID HAVING COUNT(*) > 1) AS C
  ON Tenants.TenantID = C.TenantID
ORDER BY TenantName
""")
assert rows == [("Ana",), ("Cleo",)]
print(rows)

[('Ana',), ('Cleo',)]


## 14.2 Open Requests
List every building with its number of open requests. Use a **LEFT JOIN** so buildings with zero open requests still appear.

In [3]:
rows = q("""
SELECT BuildingName, COALESCE(ReqCounts.Count, 0) AS OpenCount
FROM Buildings
LEFT JOIN (
    SELECT Apartments.BuildingID, COUNT(*) AS Count
    FROM Requests JOIN Apartments ON Requests.AptID = Apartments.AptID
    WHERE Requests.Status = 'Open'
    GROUP BY Apartments.BuildingID
) AS ReqCounts ON ReqCounts.BuildingID = Buildings.BuildingID
ORDER BY Buildings.BuildingID
""")
assert rows == [("North", 2), ("South", 2), ("East", 0)]
print(rows)

[('North', 2), ('South', 2), ('East', 0)]


## 14.3 Close All Requests
Building 11 is being renovated. Close every request from its apartments.

In [4]:
db.execute("""
UPDATE Requests SET Status = 'Closed'
WHERE AptID IN (SELECT AptID FROM Apartments WHERE BuildingID = ?)
""", (11,))
assert q("SELECT RequestID, Status FROM Requests WHERE AptID IN (110, 111)") == [(3, "Closed"), (4, "Closed")]
print(q("SELECT * FROM Requests"))

[(1, 'Open', 100, 'leak'), (2, 'Closed', 100, 'paint'), (3, 'Closed', 110, 'heater'), (4, 'Closed', 111, 'door'), (5, 'Open', 101, 'window')]


## 14.4 Joins
The join types and how they differ, shown on two small tables:
- **INNER**: only rows that match on both sides.
- **LEFT OUTER**: every left row, with NULLs where the right side has no match.
- **RIGHT OUTER**: every right row (SQLite ≥ 3.39).
- **FULL OUTER**: every row from both sides.
- **CROSS**: the Cartesian product.

In [5]:
db.executescript("""
CREATE TABLE Beverage (Name TEXT, Code TEXT);
CREATE TABLE Calorie  (Code TEXT, Cal INTEGER);
INSERT INTO Beverage VALUES ('Coke', 'COCA'), ('Water', 'H2O'), ('Juice', 'OJ');
INSERT INTO Calorie  VALUES ('COCA', 140), ('H2O', 0), ('TEA', 2);
""")
for kind in ["INNER", "LEFT OUTER", "RIGHT OUTER", "FULL OUTER"]:
    print(f"{kind:12}", q(f"SELECT Name, Calorie.Code, Cal FROM Beverage {kind} JOIN Calorie ON Beverage.Code = Calorie.Code"))
print(f"{'CROSS':12}", len(q("SELECT * FROM Beverage CROSS JOIN Calorie")), "rows")
print("sqlite version:", sqlite3.sqlite_version)

INNER        [('Coke', 'COCA', 140), ('Water', 'H2O', 0)]
LEFT OUTER   [('Coke', 'COCA', 140), ('Water', 'H2O', 0), ('Juice', None, None)]
RIGHT OUTER  [('Coke', 'COCA', 140), ('Water', 'H2O', 0), (None, 'TEA', 2)]
FULL OUTER   [('Coke', 'COCA', 140), ('Water', 'H2O', 0), ('Juice', None, None), (None, 'TEA', 2)]
CROSS        9 rows
sqlite version: 3.50.4


## 14.5 Denormalization
Denormalization means adding redundant data to cut down on joins. **Pros**: faster reads, simpler queries. **Cons**: slower and more complex writes, a risk of inconsistency, and more storage. Below we denormalize `BuildingName` into `Requests`. Reads no longer need a join, but the application now has to keep the copies in sync (here with a trigger).

In [6]:
db.executescript("""
ALTER TABLE Requests ADD COLUMN BuildingName TEXT;
UPDATE Requests SET BuildingName = (
  SELECT BuildingName FROM Apartments JOIN Buildings USING (BuildingID) WHERE Apartments.AptID = Requests.AptID);
CREATE TRIGGER sync_building_name AFTER UPDATE OF BuildingName ON Buildings
BEGIN
  UPDATE Requests SET BuildingName = NEW.BuildingName
  WHERE AptID IN (SELECT AptID FROM Apartments WHERE BuildingID = NEW.BuildingID);
END;
""")
db.execute("UPDATE Buildings SET BuildingName = 'North Tower' WHERE BuildingID = 10")
assert q("SELECT DISTINCT BuildingName FROM Requests WHERE AptID IN (100, 101)") == [("North Tower",)]
print(q("SELECT RequestID, BuildingName FROM Requests"))  # no join needed

[(1, 'North Tower'), (2, 'North Tower'), (3, 'South'), (4, 'South'), (5, 'North Tower')]


## 14.6 Entity-Relationship Diagram
Draw an ER diagram for a database of companies, people, and professionals (people who work for companies).
- `Person` is a supertype, and `Professional` is a subtype that adds degree and experience.
- `Company` ↔ `Professional` is many-to-many through `WorksFor`, which carries the salary and start date.
```
Person(PersonID, Name, Phone)
   │ is-a
Professional(PersonID FK, Degree, Experience) ──< WorksFor(PersonID, CompanyID, Salary, StartDate) >── Company(CompanyID, Name, Address)
```

In [7]:
db.executescript("""
CREATE TABLE Person       (PersonID INTEGER PRIMARY KEY, Name TEXT, Phone TEXT);
CREATE TABLE Professional (PersonID INTEGER PRIMARY KEY REFERENCES Person, Degree TEXT, Experience INTEGER);
CREATE TABLE Company      (CompanyID INTEGER PRIMARY KEY, Name TEXT, Address TEXT);
CREATE TABLE WorksFor     (PersonID INTEGER REFERENCES Professional, CompanyID INTEGER REFERENCES Company,
                           Salary INTEGER, StartDate TEXT, PRIMARY KEY (PersonID, CompanyID));
INSERT INTO Person VALUES (1,'Dee','555'),(2,'Eli','556');
INSERT INTO Professional VALUES (1,'MSc',5);
INSERT INTO Company VALUES (1,'Acme','1 Road');
INSERT INTO WorksFor VALUES (1,1,120000,'2024-01-01');
""")
print(q("""SELECT Person.Name, Company.Name, Salary FROM WorksFor
           JOIN Person USING (PersonID) JOIN Company USING (CompanyID)"""))

[('Dee', 'Acme', 120000)]


## 14.7 Design Grade Database
Design a schema for students, courses, and grades, then query for the honor roll: the **top 10%** of students by GPA. Watch the ties. Using a window function (`PERCENT_RANK`, or a GPA cutoff) keeps everyone tied at the boundary.

In [8]:
import random

db.executescript("""
CREATE TABLE Students    (StudentID INTEGER PRIMARY KEY, Name TEXT);
CREATE TABLE Courses     (CourseID INTEGER PRIMARY KEY, Name TEXT, Credits INTEGER);
CREATE TABLE CourseEnrollment (StudentID INTEGER REFERENCES Students, CourseID INTEGER REFERENCES Courses,
                               Grade REAL, Term TEXT, PRIMARY KEY (StudentID, CourseID, Term));
""")
random.seed(3)
db.executemany("INSERT INTO Students VALUES (?, ?)", [(i, f"student{i}") for i in range(1, 51)])
db.executemany("INSERT INTO Courses VALUES (?, ?, ?)", [(c, f"course{c}", random.choice([3, 4])) for c in range(1, 9)])
db.executemany("INSERT INTO CourseEnrollment VALUES (?, ?, ?, 'F26')",
               [(s, c, random.choice([2.0, 2.7, 3.0, 3.3, 3.7, 4.0])) for s in range(1, 51) for c in random.sample(range(1, 9), 4)])

honor_roll = q("""
WITH GPA AS (
  SELECT StudentID, SUM(Grade * Credits) / SUM(Credits) AS GPA
  FROM CourseEnrollment JOIN Courses USING (CourseID)
  GROUP BY StudentID
),
Ranked AS (SELECT StudentID, GPA, CUME_DIST() OVER (ORDER BY GPA DESC) AS cd FROM GPA),
Cutoff AS (SELECT MIN(GPA) AS g FROM Ranked WHERE cd <= 0.10)
SELECT Name, ROUND(GPA, 3) FROM Ranked JOIN Students USING (StudentID)
WHERE GPA >= (SELECT g FROM Cutoff)
ORDER BY GPA DESC
""")
print(honor_roll)
assert 1 <= len(honor_roll) <= 10

[('student17', 3.85), ('student16', 3.733), ('student46', 3.653), ('student8', 3.629), ('student47', 3.586)]
